<a href="https://colab.research.google.com/github/Nabin-gyawali/ml_car_race/blob/main/Racing_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Environment Setup & Hyperparameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import math

# -----------------------------------------------------------------------------
# HYPERPARAMETERS & CONFIGURATION
# -----------------------------------------------------------------------------
POP_SIZE = 60           # Cars per generation
NUM_RAYS = 7            # Sensor inputs
RAY_ANGLE_SPAN = 180    # Field of view (-90 deg to +90 deg)
RAY_MAX_DIST = 150.0    # Maximum sensor vision range (units)

MAX_SPEED = 5.0         # Maximum car speed
ACCEL_RATE = 0.2        # Acceleration delta per step
STEER_RATE = 0.08       # Maximum steering angle turn per step (radians)
FRICTION = 0.96         # Speed damping factor

ELITISM_COUNT = 6       # Top cars carried forward untouched
MUTATION_RATE = 0.12    # Probability of mutating a weight
MUTATION_SCALE = 0.15   # Gaussian noise standard deviation for mutation

MAX_STEPS = 1200         # Time limit per generation (steps)

Geometry Engine (Raycasting & Track Generation)

In [ ]:
# -----------------------------------------------------------------------------
# VECTOR GEOMETRY HELPERS
# -----------------------------------------------------------------------------
def get_segment_intersection(p1, p2, p3, p4):
    """
    Calculates intersection between line segment (p1->p2) and (p3->p4).
    Returns (intersects_bool, distance_from_p1, intersection_coords).
    """
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    x4, y4 = p4

    denom = (x2 - x1) * (y3 - y4) - (y2 - y1) * (x3 - x4)
    if abs(denom) < 1e-9:
        return False, float('inf'), None

    t = ((x3 - x1) * (y3 - y4) - (y3 - y1) * (x3 - x4)) / denom
    u = ((x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)) / denom

    if 0.0 <= t <= 1.0 and 0.0 <= u <= 1.0:
        px = x1 + t * (x2 - x1)
        py = y1 + t * (y2 - y1)
        dist = np.hypot(px - x1, py - y1)
        return True, dist, np.array([px, py])

    return False, float('inf'), None


class Track:
    """Generates inner and outer track walls from center-line waypoints."""
    def __init__(self, waypoints, track_width=35.0):
        self.waypoints = np.array(waypoints)
        self.track_width = track_width
        self.inner_wall = []
        self.outer_wall = []
        self._build_walls()

    def _build_walls(self):
        n = len(self.waypoints)
        for i in range(n):
            p_prev = self.waypoints[(i - 1) % n]
            p_curr = self.waypoints[i]
            p_next = self.waypoints[(i + 1) % n]

            # Compute tangent normal vector
            tangent = p_next - p_prev
            tangent_len = np.hypot(tangent[0], tangent[1])
            if tangent_len < 1e-6:
                normal = np.array([0.0, 1.0])
            else:
                normal = np.array([-tangent[1], tangent[0]]) / tangent_len

            self.inner_wall.append(p_curr - normal * self.track_width)
            self.outer_wall.append(p_curr + normal * self.track_width)

        self.inner_wall = np.array(self.inner_wall)
        self.outer_wall = np.array(self.outer_wall)

    def get_all_wall_segments(self):
        segments = []
        n = len(self.waypoints)
        for i in range(n):
            next_i = (i + 1) % n
            segments.append((self.inner_wall[i], self.inner_wall[next_i]))
            segments.append((self.outer_wall[i], self.outer_wall[next_i]))
        return segments


def create_sample_track():
    """Creates an elliptical oval track with turns."""
    angles = np.linspace(0, 2 * np.pi, 20, endpoint=False)
    waypoints = []
    for a in angles:
        r_x = 220 + 40 * np.sin(2 * a)
        r_y = 140 + 30 * np.cos(3 * a)
        x = r_x * np.cos(a)
        y = r_y * np.sin(a)
        waypoints.append([x, y])
    return Track(waypoints, track_width=30.0)

Neural Network Architecture (Pure NumPy)

In [ ]:
# -----------------------------------------------------------------------------
# NEURAL NETWORK (GENOME)
# -----------------------------------------------------------------------------
class NeuralNetwork:
    """
    Inputs (8):  7 Normalized Raycast Distances [0, 1] + 1 Normalized Speed [0, 1]
    Hidden (12): Tanh activation
    Outputs (2): Output 0 = Steering [-1, 1], Output 1 = Acceleration/Brake [-1, 1]
    """
    def __init__(self, input_size=8, hidden_size=12, output_size=2):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.W1 = np.random.randn(input_size, hidden_size) * 0.5
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size) * 0.5
        self.b2 = np.zeros(output_size)

    def forward(self, inputs):
        h = np.tanh(np.dot(inputs, self.W1) + self.b1)
        out = np.tanh(np.dot(h, self.W2) + self.b2)
        return out

    def get_genome(self):
        """Flattens all weights and biases into a 1D vector."""
        return np.concatenate([self.W1.ravel(), self.b1.ravel(), self.W2.ravel(), self.b2.ravel()])

    def set_genome(self, genome):
        """Restores network parameters from a 1D vector."""
        idx = 0
        w1_size = self.input_size * self.hidden_size
        self.W1 = genome[idx:idx + w1_size].reshape((self.input_size, self.hidden_size))
        idx += w1_size

        b1_size = self.hidden_size
        self.b1 = genome[idx:idx + b1_size]
        idx += b1_size

        w2_size = self.hidden_size * self.output_size
        self.W2 = genome[idx:idx + w2_size].reshape((self.hidden_size, self.output_size))
        idx += w2_size

        self.b2 = genome[idx:idx + self.output_size]

    def save_to_json(self, filepath="best_car_brain.json"):
        data = {
            "input_size": self.input_size,
            "hidden_size": self.hidden_size,
            "output_size": self.output_size,
            "W1": self.W1.tolist(),
            "b1": self.b1.tolist(),
            "W2": self.W2.tolist(),
            "b2": self.b2.tolist()
        }
        with open(filepath, "w") as f:
            json.dump(data, f, indent=4)

    @staticmethod
    def load_from_json(filepath="best_car_brain.json"):
        with open(filepath, "r") as f:
            data = json.load(f)
        nn = NeuralNetwork(data["input_size"], data["hidden_size"], data["output_size"])
        nn.W1 = np.array(data["W1"])
        nn.b1 = np.array(data["b1"])
        nn.W2 = np.array(data["W2"])
        nn.b2 = np.array(data["b2"])
        return nn

Vehicle Dynamics & Raycast Sensing

In [ ]:
# -----------------------------------------------------------------------------
# CAR SIMULATION AGENT
# -----------------------------------------------------------------------------
class Car:
    def __init__(self, start_pos, start_angle):
        self.pos = np.array(start_pos, dtype=float)
        self.angle = float(start_angle)
        self.speed = 0.0
        self.alive = True
        self.brain = NeuralNetwork()

        self.next_checkpoint_idx = 1
        self.checkpoints_passed = 0
        self.fitness = 0.0
        self.history = [self.pos.copy()]

    def cast_rays(self, wall_segments):
        distances = []
        ray_angles = np.linspace(-np.radians(RAY_ANGLE_SPAN / 2),
                                 np.radians(RAY_ANGLE_SPAN / 2),
                                 NUM_RAYS)

        for rel_angle in ray_angles:
            global_angle = self.angle + rel_angle
            ray_end = self.pos + np.array([np.cos(global_angle), np.sin(global_angle)]) * RAY_MAX_DIST

            min_dist = RAY_MAX_DIST
            for w_start, w_end in wall_segments:
                hit, dist, _ = get_segment_intersection(self.pos, ray_end, w_start, w_end)
                if hit and dist < min_dist:
                    min_dist = dist

            distances.append(min_dist / RAY_MAX_DIST)  # Normalized [0, 1]
        return np.array(distances)

    def update(self, wall_segments, track):
        if not self.alive:
            return

        # 1. Sense environment
        sensor_inputs = self.cast_rays(wall_segments)
        speed_input = np.array([self.speed / MAX_SPEED])
        network_input = np.concatenate([sensor_inputs, speed_input])

        # 2. Get steering and throttle outputs
        steering, throttle = self.brain.forward(network_input)

        # 3. Apply physics dynamics
        self.angle += steering * STEER_RATE
        self.speed += throttle * ACCEL_RATE
        self.speed = np.clip(self.speed, -MAX_SPEED * 0.3, MAX_SPEED)
        self.speed *= FRICTION

        self.pos += np.array([np.cos(self.angle), np.sin(self.angle)]) * self.speed
        self.history.append(self.pos.copy())

        # 4. Check for wall collision
        for w_start, w_end in wall_segments:
            # Check if car position crossed any wall
            hit, dist, _ = get_segment_intersection(self.pos, self.pos - np.array([np.cos(self.angle), np.sin(self.angle)]) * self.speed, w_start, w_end)
            if hit or np.min(sensor_inputs) < (8.0 / RAY_MAX_DIST):
                self.alive = False
                break

        # 5. Check checkpoint progress
        num_waypoints = len(track.waypoints)
        target_wp = track.waypoints[self.next_checkpoint_idx % num_waypoints]
        dist_to_target = np.hypot(self.pos[0] - target_wp[0], self.pos[1] - target_wp[1])

        if dist_to_target < track.track_width * 1.2:
            self.checkpoints_passed += 1
            self.next_checkpoint_idx = (self.next_checkpoint_idx + 1) % num_waypoints

        # Calculate total fitness score
        self.fitness = (self.checkpoints_passed * 1000.0) - dist_to_target + (len(self.history) * 0.1)

Genetic Algorithm (Selection & Breeding)

In [ ]:
# -----------------------------------------------------------------------------
# GENETIC ALGORITHM ENGINE
# -----------------------------------------------------------------------------
class GeneticAlgorithm:
    @staticmethod
    def breed_next_generation(population, start_pos, start_angle):
        # Sort cars by fitness (descending)
        population.sort(key=lambda car: car.fitness, reverse=True)

        new_population = []

        # 1. Elitism: Preserve top performers
        for i in range(ELITISM_COUNT):
            elite_car = Car(start_pos, start_angle)
            elite_car.brain.set_genome(population[i].brain.get_genome().copy())
            new_population.append(elite_car)

        # 2. Tournament selection & crossover
        def select_parent():
            candidates = np.random.choice(population[:20], size=3, replace=False)
            candidates = list(candidates)
            candidates.sort(key=lambda c: c.fitness, reverse=True)
            return candidates[0]

        while len(new_population) < POP_SIZE:
            parent_a = select_parent()
            parent_b = select_parent()

            genome_a = parent_a.brain.get_genome()
            genome_b = parent_b.brain.get_genome()

            # Single-point crossover
            cross_point = np.random.randint(0, len(genome_a))
            child_genome = np.concatenate([genome_a[:cross_point], genome_b[cross_point:]])

            # Mutation step
            for g in range(len(child_genome)):
                if np.random.rand() < MUTATION_RATE:
                    child_genome[g] += np.random.normal(0, MUTATION_SCALE)

            child_car = Car(start_pos, start_angle)
            child_car.brain.set_genome(child_genome)
            new_population.append(child_car)

        return new_population

Training Runner & Visualization (Colab Output)

In [ ]:
# -----------------------------------------------------------------------------
# MAIN TRAINING LOOP & VISUALIZATION
# -----------------------------------------------------------------------------
def run_neuroevolution(generations=100):
    track = create_sample_track()
    wall_segments = track.get_all_wall_segments()

    # Determine initial car direction towards waypoint 1
    start_pos = track.waypoints[0]
    next_wp = track.waypoints[1]
    start_angle = math.atan2(next_wp[1] - start_pos[1], next_wp[0] - start_pos[0])

    population = [Car(start_pos, start_angle) for _ in range(POP_SIZE)]
    best_overall_car = None

    print(f"--- Starting Neuroevolution on Track ({generations} Generations) ---")

    for gen in range(1, generations + 1):
        # Reset generation state
        for car in population:
            car.pos = np.array(start_pos, dtype=float)
            car.angle = float(start_angle)
            car.speed = 0.0
            car.alive = True
            car.history = [car.pos.copy()]
            car.checkpoints_passed = 0
            car.next_checkpoint_idx = 1

        # Simulate steps until time limit or all cars crash
        for step in range(MAX_STEPS):
            active_cars = [c for c in population if c.alive]
            if not active_cars:
                break
            for car in active_cars:
                car.update(wall_segments, track)

        # Identify generation leader
        population.sort(key=lambda c: c.fitness, reverse=True)
        gen_best = population[0]

        if best_overall_car is None or gen_best.fitness > best_overall_car.fitness:
            best_overall_car = gen_best

        print(f"Gen {gen:02d}/{generations:02d} | Best Fitness: {gen_best.fitness:8.1f} | Checkpoints Passed: {gen_best.checkpoints_passed}")

        # Evolve population for next generation
        if gen < generations:
            population = GeneticAlgorithm.breed_next_generation(population, start_pos, start_angle)

    # Save top performer
    best_overall_car.brain.save_to_json("best_car_brain.json")
    print("\nBest car brain saved to 'best_car_brain.json' successfully.")

    # Render Visual Summary
    plt.figure(figsize=(10, 7))
    plt.plot(track.inner_wall[:, 0], track.inner_wall[:, 1], 'k-', linewidth=2, label="Inner Wall")
    plt.plot(track.outer_wall[:, 0], track.outer_wall[:, 1], 'k-', linewidth=2, label="Outer Wall")

    # Plot driving line of the top performer
    path = np.array(best_overall_car.history)
    plt.plot(path[:, 0], path[:, 1], 'r-', linewidth=2.5, label="Best AI Driving Trajectory")
    plt.scatter([start_pos[0]], [start_pos[1]], color='green', s=100, zorder=5, label="Start Line")

    plt.title(f"Neuroevolution Results - Best Checkpoints: {best_overall_car.checkpoints_passed}")
    plt.axis("equal")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(loc="lower right")
    plt.show()

# Run training
run_neuroevolution(generations=25)

--- Starting Neuroevolution on Track (25 Generations) ---
Gen 01/25 | Best Fitness:  44081.2 | Checkpoints Passed: 44
Gen 02/25 | Best Fitness:  69081.6 | Checkpoints Passed: 69
Gen 03/25 | Best Fitness:  69081.6 | Checkpoints Passed: 69
Gen 04/25 | Best Fitness:  77082.6 | Checkpoints Passed: 77
Gen 05/25 | Best Fitness:  83021.0 | Checkpoints Passed: 83
Gen 06/25 | Best Fitness:  87060.8 | Checkpoints Passed: 87
Gen 07/25 | Best Fitness:  89074.6 | Checkpoints Passed: 89
Gen 08/25 | Best Fitness:  91066.1 | Checkpoints Passed: 91
Gen 09/25 | Best Fitness:  93080.9 | Checkpoints Passed: 93
Gen 10/25 | Best Fitness:  95076.3 | Checkpoints Passed: 95
Gen 11/25 | Best Fitness:  97052.0 | Checkpoints Passed: 97
Gen 12/25 | Best Fitness:  98073.8 | Checkpoints Passed: 98
Gen 13/25 | Best Fitness: 100087.0 | Checkpoints Passed: 100
